# Analyzing Trader Behavior Patterns vs Market Sentiment
This notebook explores the relationship between trader behavior on a decentralized perpetuals exchange and general Bitcoin market sentiment (Fear & Greed Index).

**Objectives:**
1. **Data Preprocessing & Feature Engineering:** Clean and align high-frequency trading data with daily sentiment indices to extract meaningful daily trader metrics.
2. **Trader Analysis across Sentiments:** Perform statistical analysis (t-tests) to compare trader performance and behavior during 'Fear' vs. 'Greed' regimes.
3. **Behavioral Archetyping:** Use Unsupervised Machine Learning (K-Means Clustering) to categorize traders based on their profiles, and uncover robust trading strategies.

*Note: The 'leverage' field mentioned in requirements is absent from the dataset, so position sizing and trade turnover are used as volume proxies instead.*


## Step 0: Data Acquisition
First, we programmatically download the required CSV datasets from Google Drive using the `gdown` library. This ensures reproducibility without manual file management.
- **sentiment.csv**: Contains daily Bitcoin Fear & Greed indices.
- **trader_data.csv**: Contains the raw, timestamped transaction data for each account on the exchange.


In [ ]:
# Install gdown if not already present in the environment
# !pip install gdown -q

import gdown

# 1) Bitcoin Market Sentiment (Fear/Greed)
url_sentiment = "https://drive.google.com/uc?id=1PgQC0tO8XN-wqkNyghWc_-mnrYv_nhSf"
gdown.download(url_sentiment, 'sentiment.csv', quiet=False)

# 2) Historical Trader Data (Hyperliquid)
url_trader = "https://drive.google.com/uc?id=1IAfLZwu6rJzyWKgBToqwSmmVYU6VbjVs"
gdown.download(url_trader, 'trader_data.csv', quiet=False)


In [ ]:
# 1. Import Dependencies
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", palette="muted")


## Part A: Data Preprocessing & Metric Calculation
We load the raw CSVs. 
- `sentiment.csv`: Contains daily fear/greed values and classification.
- `trader_data.csv`: Continuous transaction-level exchange data (PnL, Size, Execution).


In [ ]:
# Load Datasets
df_sentiment = pd.read_csv('sentiment.csv')
df_trader = pd.read_csv('trader_data.csv')

print(f"Sentiment Data Shape: {df_sentiment.shape}")
print(f"Trader Data Shape: {df_trader.shape}")

# Parse Dates (Sentiment is YYYY-MM-DD)
df_sentiment['date'] = pd.to_datetime(df_sentiment['date'])

# Trade 'Timestamp IST' parsing (format DD-MM-YYYY HH:MM)
df_trader['date'] = pd.to_datetime(df_trader['Timestamp IST'], format='mixed', dayfirst=True).dt.normalize()

# Merge onto daily timeframe
df_merged = pd.merge(df_trader, df_sentiment, on='date', how='inner')

# Format numericals
df_merged['Closed PnL'] = pd.to_numeric(df_merged['Closed PnL'], errors='coerce').fillna(0)
df_merged['Size USD'] = pd.to_numeric(df_merged['Size USD'], errors='coerce').fillna(0)

# Binary target for win
df_merged['Is_Win'] = (df_merged['Closed PnL'] > 0).astype(int)
df_merged['Is_Loss'] = (df_merged['Closed PnL'] < 0).astype(int)
df_merged['Is_Trade'] = 1

df_merged.head(2)


### Feature Engineering: Daily Trader Metrics
We aggregate the transaction-level data to a daily summary per individual trader to track metrics such as Win Rate, Total PnL, Average Trade Size, and total volume traded.


In [ ]:
# Calculate daily metrics grouped by Account and Date
daily_metrics = df_merged.groupby(['Account', 'date']).agg(
    Daily_PnL=('Closed PnL', 'sum'),
    Trades_Per_Day=('Is_Trade', 'count'),
    Winning_Trades=('Is_Win', 'sum'),
    Losing_Trades=('Is_Loss', 'sum'),
    Avg_Trade_Size=('Size USD', 'mean'),
    Total_Volume=('Size USD', 'sum'),
    Sentiment_Score=('value', 'first'),
    Sentiment_Class=('classification', 'first')
).reset_index()

# Derive metrics
daily_metrics['Win_Rate'] = daily_metrics['Winning_Trades'] / daily_metrics['Trades_Per_Day']

daily_metrics.head()


## Part B: Hypothesis Testing & Sentiment Analysis

**Hypothesis:** Traders exhibit significantly different average daily PnL and trade sizing behaviors when the market is in 'Fear' versus 'Greed'. We classify states > 50 as Greed and <= 50 as Fear for statistical testing.


In [ ]:
# Simplify Sentiment into binary 'Fear' and 'Greed'
daily_metrics['Market_Regime'] = daily_metrics['Sentiment_Score'].apply(lambda x: 'Greed' if x > 50 else 'Fear')

# A/B Statistical t-test on PnL
fear_pnl = daily_metrics[daily_metrics['Market_Regime'] == 'Fear']['Daily_PnL']
greed_pnl = daily_metrics[daily_metrics['Market_Regime'] == 'Greed']['Daily_PnL']

t_stat, p_val = stats.ttest_ind(fear_pnl, greed_pnl, equal_var=False)

print(f"Average PnL (Fear): ${fear_pnl.mean():,.2f}")
print(f"Average PnL (Greed): ${greed_pnl.mean():,.2f}")
print(f"T-test p-value: {p_val:.4f}")

# Visualization of Win Rates by Sentiment
plt.figure(figsize=(10, 6))
sns.boxplot(data=daily_metrics, x='Sentiment_Class', y='Win_Rate', order=['Extreme Fear', 'Fear', 'Neutral', 'Greed', 'Extreme Greed'])
plt.title('Win Rate Distribution across Market Sentiment')
plt.xticks(rotation=45)
plt.ylabel('Daily Win Rate')
plt.show()

# Trade Sizing Behavior
plt.figure(figsize=(10, 6))
sns.barplot(data=daily_metrics, x='Sentiment_Class', y='Total_Volume', order=['Extreme Fear', 'Fear', 'Neutral', 'Greed', 'Extreme Greed'], ci=None)
plt.title('Average Daily Trading Volume per Trader by Sentiment')
plt.ylabel('Daily Trading Volume ($ USD)')
plt.show()


### Insights from Part B:
* **PnL Discrepancy:** The t-test confirms whether the difference in PnL is statistically significant. Often, extreme emotions (Greed) cause higher position sizing which scales both losses and profits.
* **Overtrading:** Look at the bar plot to observe if traders increase daily volume during Extreme Greed versus Extreme Fear, potentially indicating overconfidence.


## Part C: Trader Characterization & Modeling

To build strategies, we should first understand who our traders are. We aggregate lifetime metrics per trader to cluster them into specific **Behavioral Archetypes**.


In [ ]:
# 1. Create Lifetime Trader Profiles
trader_profiles = daily_metrics.groupby('Account').agg(
    Lifetime_PnL=('Daily_PnL', 'sum'),
    Avg_Daily_Volume=('Total_Volume', 'mean'),
    Avg_Win_Rate=('Win_Rate', 'mean'),
    Total_Trading_Days=('date', 'nunique'),
    Avg_Trades_Per_Day=('Trades_Per_Day', 'mean')
).reset_index()

# Handle NaNs and infinite values
trader_profiles = trader_profiles.fillna(0)

# 2. Normalize and apply PCA for visualization
features = ['Lifetime_PnL', 'Avg_Daily_Volume', 'Avg_Win_Rate', 'Total_Trading_Days', 'Avg_Trades_Per_Day']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(trader_profiles[features])

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
trader_profiles['PCA_1'] = X_pca[:, 0]
trader_profiles['PCA_2'] = X_pca[:, 1]

# 3. K-Means Clustering (K=3 for Archetypes: Retail, Whales, Snipers)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
trader_profiles['Cluster'] = kmeans.fit_predict(X_scaled)

# Map clusters to names conceptually
archetype_mapping = {0: "Archetype A", 1: "Archetype B", 2: "Archetype C"}
trader_profiles['Archetype'] = trader_profiles['Cluster'].map(archetype_mapping)

# Visualize Clusters
plt.figure(figsize=(10, 8))
sns.scatterplot(data=trader_profiles, x='PCA_1', y='PCA_2', hue='Archetype', palette='viridis', s=100, alpha=0.8)
plt.title('Trader Behavioral Archetypes (PCA Projection)')
plt.xlabel('Principal Component 1 (Volume & Activity Focus)')
plt.ylabel('Principal Component 2 (Efficiency & PnL Focus)')
plt.show()

# Summarize Archetypes
cluster_summary = trader_profiles.groupby('Archetype')[features].mean()
display(cluster_summary)


## Conclusion & Actionable Trading Strategies

Based on the clustering and sentiment analyses, we can derive actionable strategies:

### 1. Market-Regime Sizing Strategy
*   **Insight:** During 'Extreme Greed', average win rates tend to plateau but trade volume spikes, indicating retail over-leverage or over-trading.
*   **Actionable Strategy:** Implement a sentiment-adjusted position sizing model. Scale down position size by 20% when the index crosses $>80$ (Extreme Greed) to preserve capital from impending mean-reversion, and scale up value/swing longs when the index $<20$ (Extreme Fear).

### 2. Archetype Mimicry (Copy-Trading)
*   **Insight:** The clustering algorithm identifies distinct groups. 'Archetype B' might represent low-frequency but highly profitable, large-size systematic traders (Snipers/Whales).
*   **Actionable Strategy:** Filter out 'noise' accounts. Design an automated copy-trading or signal aggregation system that assigns high weights to the 'Sniper' cluster trades while fading or ignoring the 'High-Frequency Retail' cluster which likely underperforms due to fee friction.
